# GOaT selection notebook

Run top to bottom on Colab. Every stage writes its artifacts to Google Drive
and skips itself when those artifacts already exist.

## Step 1. Open the Colab notebook

## Step 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Step 3. Git clone the repo, shallow

In [ ]:
![ -d /content/GOaT/.git ] || git clone --depth 1 https://github.com/champyod/GOAT.git /content/GOaT

## Step 4. Install dependencies

In [ ]:
%pip install -q transformers datasets sentencepiece peft accelerate jiwer sacrebleu opencv-python-headless scipy psutil tqdm matplotlib paddleocr paddlepaddle

## Step 5. Import the package from the clone

In [ ]:
import sys

sys.path.insert(0, "/content/GOaT/model/src")

from goat_model import constants as c

## Step 6. Stage datasets, Drive checked, MT and OCR selection

In [ ]:
from pathlib import Path

from goat_model.data import (
    download_flores200,
    download_scbmt,
    download_thaiocr_evaluation,
    download_thaiocrbench,
)

DATA_ROOT = Path("/content/drive/MyDrive/GOaT/datasets")
MT_DIR = DATA_ROOT / "mt"
OCR_DIR = DATA_ROOT / "ocr"

if not (MT_DIR / "train.th").is_file():
    download_scbmt(MT_DIR)

if not (MT_DIR / "test" / "flores200.th").is_file():
    download_flores200(MT_DIR / "test")

if not (OCR_DIR / "thai-ocr-evaluation" / "images").is_dir():
    download_thaiocr_evaluation(OCR_DIR / "thai-ocr-evaluation")

if not (OCR_DIR / "thaiocrbench" / "images").is_dir():
    download_thaiocrbench(OCR_DIR / "thaiocrbench")

## Step 7. MT model selection

Methodology, model selection MT. Both NLLB candidates zero shot EN->TH on
FLORES-200, batch 16, 5 repeats. Rule, NLLB-600M iff BLEU > 35 and latency
<= 2 s. Result lands in results/mt_selection.json on Drive.

In [ ]:
from pathlib import Path

from goat_model.constants import (
    LANG_CODES,
    MT_BATCH_SIZE,
    MT_BEAM_SIZE,
    MT_BLEU_THRESHOLD,
    MT_LATENCY_THRESHOLD_S,
    MT_MAX_LENGTH,
    MT_MODELS,
    MT_N_RUNS,
    SEED,
)
from goat_model.metrics import paired_t_test, summarize
from goat_model.mt.engine import get_mt
from goat_model.mt.evaluate import load_pairs, run_mt
from goat_model.utils import setup_seed, write_json

setup_seed(SEED)

MT_SELECTION = Path("/content/drive/MyDrive/GOaT/results/mt_selection.json")

if MT_SELECTION.is_file():
    print(f"skipped - already selected: {MT_SELECTION}")
else:
    src, refs, _ = load_pairs(MT_DIR / "test" / "flores200.en", MT_DIR / "test" / "flores200.th")

    bleu_runs, lat_runs = {}, {}
    for model in MT_MODELS:
        backend = get_mt(model, LANG_CODES["en"], LANG_CODES["th"],
                         beam=MT_BEAM_SIZE, max_length=MT_MAX_LENGTH)
        bleu_runs[model], lat_runs[model] = [], []
        for _ in range(MT_N_RUNS):
            r = run_mt(backend, src, refs, batch_size=MT_BATCH_SIZE)
            bleu_runs[model].append(r["bleu"])
            lat_runs[model].append(r["average_ms_per_sentence"] / 1000)

    m600 = summarize(bleu_runs["NLLB-200-distilled-600M"])[0]
    l600 = summarize(lat_runs["NLLB-200-distilled-600M"])[0]
    selected = ("NLLB-200-distilled-600M"
                if m600 > MT_BLEU_THRESHOLD and l600 <= MT_LATENCY_THRESHOLD_S
                else "NLLB-200-distilled-1.3B")

    write_json(
        MT_SELECTION,
        {
            "models": {
                m: {"bleu": summarize(bleu_runs[m]), "avg_s_per_sent": summarize(lat_runs[m])}
                for m in MT_MODELS
            },
            "paired_t_test": paired_t_test(bleu_runs[MT_MODELS[0]], bleu_runs[MT_MODELS[1]]),
            "decision_rule": f"{MT_MODELS[0]} iff BLEU > {MT_BLEU_THRESHOLD} "
                             f"and latency <= {MT_LATENCY_THRESHOLD_S}s",
            "selected": selected,
        },
    )

## Step 8. OCR model selection

Methodology, model selection OCR. Both candidates on both public benchmark
sets, CER and latency, 5 repeats. Rule, ThaiTrOCR iff mean CER <= 0.10,
else lowest CER. Result lands in results/ocr_selection.json on Drive.

In [ ]:
from pathlib import Path

from goat_model.constants import (
    OCR_ALPHA,
    OCR_CER_THRESHOLD,
    OCR_DATASETS,
    OCR_IMG_SIZE,
    OCR_MODELS,
    OCR_N_RUNS,
    SEED,
)
from goat_model.metrics import cohens_d, paired_t_test
from goat_model.ocr.engine import get_ocr
from goat_model.ocr.evaluate import aggregate_records, discover_assets, run_ocr
from goat_model.utils import setup_seed, write_json

setup_seed(SEED)

OCR_SELECTION = Path("/content/drive/MyDrive/GOaT/results/ocr_selection.json")

if OCR_SELECTION.is_file():
    print(f"skipped - already selected: {OCR_SELECTION}")
else:
    per_dataset, cer_flat = {}, {m: [] for m in OCR_MODELS}
    for model in OCR_MODELS:
        per_dataset[model] = {}
        backend = get_ocr(model)
        for dataset in OCR_DATASETS:
            assets = discover_assets(OCR_DIR / dataset)
            runs = [run_ocr(backend, assets, OCR_IMG_SIZE[model]) for _ in range(OCR_N_RUNS)]
            per_dataset[model][dataset] = aggregate_records(runs)
            cer_flat[model] += [rec["cer"] for run in runs for rec in run]

    thai_mean = sum(cer_flat["ThaiTrOCR"]) / len(cer_flat["ThaiTrOCR"])
    pp_mean = sum(cer_flat["PP-OCRv5-mobile"]) / len(cer_flat["PP-OCRv5-mobile"])
    selected_ocr = ("ThaiTrOCR" if thai_mean <= OCR_CER_THRESHOLD
                    else ("ThaiTrOCR" if thai_mean < pp_mean else "PP-OCRv5-mobile"))

    write_json(
        OCR_SELECTION,
        {
            "n_runs": OCR_N_RUNS,
            "models": per_dataset,
            "comparisons": [{
                "a": "ThaiTrOCR", "b": "PP-OCRv5-mobile",
                "paired_t_test": paired_t_test(
                    cer_flat["ThaiTrOCR"], cer_flat["PP-OCRv5-mobile"], alpha=OCR_ALPHA
                ),
                "cohens_d": cohens_d(cer_flat["ThaiTrOCR"], cer_flat["PP-OCRv5-mobile"]),
            }],
            "decision_rule": f"ThaiTrOCR iff mean CER <= {OCR_CER_THRESHOLD}, else lowest CER",
            "mean_cer_thaitrocr": thai_mean,
            "mean_cer_ppocrv5": pp_mean,
            "selected": selected_ocr,
        },
    )